In [ ]:
# --- Bloco 1: Setup do Ambiente e Upload do Arquivo ---

# Exibe uma mensagem para o usuário, informando o início da verificação do ambiente.
print("--- Verificando e Importando Bibliotecas ---");

# Importa bibliotecas necessárias para interagir com o sistema e executar comandos.
import sys;
import subprocess;

# Importar a biblioteca matplotlib
try:
    __import__('matplotlib');
    print("✔️  Biblioteca 'matplotlib' já está instalada.");
except ImportError:
    print("⚠️  Biblioteca 'matplotlib' não encontrada. Instalando...");
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 'matplotlib']);
    print("✅  Biblioteca 'matplotlib' instalada com sucesso!");

# Importações principais para a funcionalidade do script
try:
    from google.colab import files; # Específico do Google Colab, para habilitar o upload de arquivos
    in_colab = True
except ImportError:
    print("⚠️  Biblioteca 'google.colab' não encontrada. Esta funcionalidade só está disponível no Google Colab.");
    in_colab = False

import matplotlib.pyplot as plt; # Biblioteca para criação de gráficos e visualizações
import os; # Fornece funções para interagir com o sistema operacional

print("\nAmbiente pronto para a próxima etapa!");

In [ ]:
# --- Bloco 2: Definição das Funções e Carregamento dos Dados ---

# Variável global para armazenar os dados lidos do arquivo CSV.
dados_meteorologicos = None;

def processar_data(data_str):
    try:
        if '/' in data_str:
            # Divide a string pelo caractere '/' e retorna o ano e o mês.
            partes = data_str.split('/');
            return int(partes[2]), int(partes[1]);
        elif '-' in data_str:
            # Divide a string pelo caractere '-' e retorna o ano e o mês.
            partes = data_str.split('-');
            return int(partes[0]), int(partes[1]);
    except (ValueError, IndexError):
        # Captura erros se a conversão para inteiro falhar ou se a data não tiver partes suficientes.
        return None;
    return None;

def carregar_dados(arquivo):
    print(f"\nIniciando carregamento de '{arquivo}'...");
    dados = [];

    try:
        with open(arquivo, 'r', encoding='utf-8') as f:
            next(f);  # Pula a primeira linha (cabeçalho).
            # Itera sobre cada linha
            for num_linha, linha in enumerate(f, start=2):
                linha_limpa = linha.strip();
                if not linha_limpa:
                  continue;  # Ignora linhas completamente vazias

                dados_linha = linha_limpa.split(',');
                # Validação: verifica se a linha tem colunas suficientes e uma data válida
                if len(dados_linha) < 6 or processar_data(dados_linha[0]) is None:
                    print(f"Aviso: Linha {num_linha} mal formatada ou com data inválida. Ignorando.");
                    continue;
                try:
                    # Adiciona a linha processada à lista de dados
                    dados.append([
                        dados_linha[0],
                        float(dados_linha[1]) if dados_linha[1] else 0.0,
                        float(dados_linha[2]) if dados_linha[2] else 0.0,
                        float(dados_linha[3]) if dados_linha[3] else 0.0,
                        float(dados_linha[4]) if dados_linha[4] else 0.0,
                        float(dados_linha[5]) if dados_linha[5] else 0.0
                    ]);
                except ValueError:
                    # Captura erro se algum dado que deveria ser numérico não puder ser convertido
                    print(f"Aviso: Linha {num_linha} com dados numéricos inválidos. Ignorando.");
                    continue
        print(f"Sucesso! {len(dados)} registros válidos foram carregados.");
        return dados;
    except Exception as e:
        # Captura erros mais graves, como arquivo não encontrado ou problemas de permissão
        print(f"ERRO CRÍTICO ao processar o arquivo: {e}");
        return None;

def visualizar_intervalo_dados(dados):
    print("\n--- Visualização de Dados por Período ---");

    try:
        # Determina o intervalo de anos disponíveis nos dados para informar o usuário.
        primeiro_ano = processar_data(dados[0][0])[0];
        ultimo_ano = processar_data(dados[-1][0])[0];
    except (TypeError, IndexError):
        print("ERRO: Não foi possível determinar o intervalo de datas dos dados.");
        return;

    print(f"INFO: Os dados disponíveis cobrem o período de {primeiro_ano} a {ultimo_ano}.");

    # Coleta e valida a entrada do usuário para o período desejado
    while True:
        try:
            ano_inicio = int(input(f"Informe o ano inicial ({primeiro_ano}-{ultimo_ano}): "));
            if primeiro_ano <= ano_inicio <= ultimo_ano:
              break;
            else:
              print(f"ERRO: Ano fora do intervalo permitido.");
        except ValueError:
          print("ERRO: Entrada inválida. Digite um número.");
    while True:
        try:
            mes_inicio = int(input("Informe o mês inicial (1-12): "));
            if 1 <= mes_inicio <= 12:
              break;
            else:
              print("ERRO: Mês inválido.");
        except ValueError:
          print("ERRO: Entrada inválida.");
    while True:
        try:
            ano_fim = int(input(f"Informe o ano final ({ano_inicio}-{ultimo_ano}): "));
            if ano_inicio <= ano_fim <= ultimo_ano:
              break;
            else:
              print(f"ERRO: Ano inválido. Deve ser entre {ano_inicio} e {ultimo_ano}.");
        except ValueError: print("ERRO: Entrada inválida.");
    while True:
        try:
            mes_fim = int(input("Informe o mês final (1-12): "));
            if (ano_inicio, mes_inicio) <= (ano_fim, mes_fim):
              break;
            else:
              print("ERRO: O mês/ano final não pode ser anterior ao inicial.");
        except ValueError:
          print("ERRO: Entrada inválida.");

    # Pergunta ao usuário qual conjunto de dados ele quer ver.
    print("\nQual tipo de dado você deseja visualizar?\n1) Todos os dados\n2) Apenas precipitação\n3) Apenas temperaturas\n4) Apenas umidade e vento");
    while True:
        try:
            escolha = int(input("Escolha uma opção (1-4): "));
            if escolha in [1, 2, 3, 4]:
              break;
            else:
              print("ERRO: Opção inválida.");
        except ValueError:
          print("ERRO: Entrada inválida.");

    # Variáveis para formatação da tabela
    w_data, w_precip, w_tmax, w_tmin, w_umid, w_vento = 12, 16, 14, 14, 12, 13;
    h_data, h_precip, h_tmax, h_tmin, h_umid, h_vento = "Data", "Precip(mm)", "T.Max(C)", "T.Min(C)", "Umid(%)", "Vento(m/s)";

    # Constrói e imprime o cabeçalho da tabela
    if escolha == 1:
        div = f"+{'-'*(w_data+2)}+{'-'*(w_precip+2)}+{'-'*(w_tmax+2)}+{'-'*(w_tmin+2)}+{'-'*(w_umid+2)}+{'-'*(w_vento+2)}+";
        print(f"\n{div}\n| {h_data:<{w_data}} | {h_precip:^{w_precip}} | {h_tmax:^{w_tmax}} | {h_tmin:^{w_tmin}} | {h_umid:^{w_umid}} | {h_vento:^{w_vento}} |\n{div}");
    elif escolha == 2:
        div = f"+{'-'*(w_data+2)}+{'-'*(w_precip+2)}+"; print(f"\n{div}\n| {h_data:<{w_data}} | {h_precip:^{w_precip}} |\n{div}");
    elif escolha == 3:
        div = f"+{'-'*(w_data+2)}+{'-'*(w_tmax+2)}+{'-'*(w_tmin+2)}+"; print(f"\n{div}\n| {h_data:<{w_data}} | {h_tmax:^{w_tmax}} | {h_tmin:^{w_tmin}} |\n{div}");
    elif escolha == 4:
        div = f"+{'-'*(w_data+2)}+{'-'*(w_umid+2)}+{'-'*(w_vento+2)}+"; print(f"\n{div}\n| {h_data:<{w_data}} | {h_umid:^{w_umid}} | {h_vento:^{w_vento}} |\n{div}");

    dados_encontrados = 0
    # Itera sobre todos os registros para encontrar e exibir os que correspondem ao filtro.
    for registro in dados:
        data_proc = processar_data(registro[0]);
        if data_proc:
            ano_registro, mes_registro = data_proc;
            # Verifica se o registro está dentro do intervalo de data solicitado.
            if (ano_inicio, mes_inicio) <= (ano_registro, mes_registro) <= (ano_fim, mes_fim):
                dados_encontrados += 1;
                # Imprime a linha de dados formatada de acordo com a escolha do usuário.
                if escolha == 1:
                  print(f"| {registro[0]:<{w_data}} | {f'{registro[1]:.1f} mm':>{w_precip}} | {f'{registro[2]:.1f} C':>{w_tmax}} | {f'{registro[3]:.1f} C':>{w_tmin}} | {f'{registro[4]:.0f}%':>{w_umid}} | {f'{registro[5]:.1f} m/s':>{w_vento}} |");
                elif escolha == 2:
                  print(f"| {registro[0]:<{w_data}} | {f'{registro[1]:.1f} mm':>{w_precip}} |");
                elif escolha == 3:
                  print(f"| {registro[0]:<{w_data}} | {f'{registro[2]:.1f} C':>{w_tmax}} | {f'{registro[3]:.1f} C':>{w_tmin}} |");
                elif escolha == 4:
                  print(f"| {registro[0]:<{w_data}} | {f'{registro[4]:.0f}%':>{w_umid}} | {f'{registro[5]:.1f} m/s':>{w_vento}} |");

    # Finaliza a tabela ou informa que nenhum dado foi encontrado.
    if dados_encontrados > 0:
      print(div);
    else:
      print("Nenhum dado encontrado para o período especificado.");

def encontrar_mes_chuvoso(dados):
    print("\n--- Análise: Mês Mais Chuvoso (Todo o Período) ---")

    # Dicionário para acumular a chuva
    chuva_por_mes = {}
    for registro in dados:
        data_processada = processar_data(registro[0])
        if data_processada:
            ano, mes = data_processada
            chave_mes_ano = f"{ano}-{mes:02d}"
            chuva_por_mes[chave_mes_ano] = chuva_por_mes.get(chave_mes_ano, 0) + registro[1]

    if not chuva_por_mes:
        print("Não foi possível analisar dados de chuva.")
        return

    # Encontra o valor máximo de chuva e o(s) mês(es) correspondente(s).
    maior_chuva = -1
    meses_campeoes_chave = [] # Armazena as chaves
    for mes_chave, total in chuva_por_mes.items():
        if total > maior_chuva:
            maior_chuva = total
            meses_campeoes_chave = [mes_chave] # Novo recorde, reinicia a lista.
        elif total == maior_chuva:
            meses_campeoes_chave.append(mes_chave) # Empate, adiciona à lista.

    # Lista de nomes dos meses para conversão
    nomes_dos_meses = [
        "", "Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
        "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro"
    ]

    # Lista para armazenar os nomes formatados por extenso
    meses_formatados = []
    for chave in meses_campeoes_chave:
        # Quebra a chave 'YYYY-MM'
        ano, mes_num = int(chave[:4]), int(chave[5:])
        # Formata a string com o nome do mês
        nome_formatado = f"{nomes_dos_meses[mes_num]} de {ano}"
        meses_formatados.append(nome_formatado)

    # Exibe o resultado, tratando o caso de um único vencedor ou de múltiplos.
    if len(meses_formatados) == 1:
        print(f"O mês mais chuvoso foi {meses_formatados[0]} com {maior_chuva:.2f} mm.")
    else:
        print(f"Os meses mais chuvosos foram: {', '.join(meses_formatados)}")
        print(f"Com um volume de {maior_chuva:.2f} mm cada.")

def analisar_temperatura_minima(dados):
    print("\n--- Análise de Temperatura Mínima por Mês (Período: 2006-2016) ---");

    try:
        mes_escolhido = int(input("Informe o mês para análise (1-12): "));
        if not 1 <= mes_escolhido <= 12:
            print("ERRO: Mês inválido.");
            return;
    except ValueError:
        print("ERRO: Entrada inválida.");
        return;

    # Dicionários para armazenar a soma das temperaturas e a contagem de dias por ano.
    somas_temp_por_ano = {};
    contagens_dias_por_ano = {};

    for registro in dados:
        data_proc = processar_data(registro[0]);
        if data_proc:
            ano, mes = data_proc;
            # Filtra os dados para o período e mês de interesse.
            if 2006 <= ano <= 2016 and mes == mes_escolhido:
                ano_str = str(ano);
                somas_temp_por_ano[ano_str] = somas_temp_por_ano.get(ano_str, 0) + registro[3];
                contagens_dias_por_ano[ano_str] = contagens_dias_por_ano.get(ano_str, 0) + 1;

    if not somas_temp_por_ano:
        print(f"Não há dados para o mês {mes_escolhido} entre 2006 e 2016.");
        return;

    nomes_dos_meses = ["", "Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho", "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro"];
    nome_mes = nomes_dos_meses[mes_escolhido];

    # Calcula e exibe a média de temperatura mínima para cada ano no período.
    medias_anuais = {};
    print(f"\nMédias de temperatura mínima para {nome_mes}:");
    for ano, soma in sorted(somas_temp_por_ano.items()):
        media = soma / contagens_dias_por_ano[ano];
        medias_anuais[ano] = media;
        print(f"  - Ano {ano}: {media:.2f}°C");

    # Calcula e exibe a média geral para o mês em todo o período.
    media_geral = sum(medias_anuais.values()) / len(medias_anuais);
    print(f"\nMédia geral para {nome_mes} no período: {media_geral:.2f}°C");

    # Geração do gráfico de barras com matplotlib.
    print("\nGerando gráfico de barras...");
    anos = list(medias_anuais.keys());
    medias = list(medias_anuais.values());

    plt.figure(figsize=(12, 7));  # Define o tamanho da figura do gráfico.
    plt.bar(anos, medias, color='dodgerblue', edgecolor='black');  # Cria as barras.
    plt.xlabel("Ano");  # Rótulo do eixo X.
    plt.ylabel("Temperatura Mínima Média (°C)");  # Rótulo do eixo Y.
    plt.title(f"Média da Temperatura Mínima em {nome_mes} (2006-2016)");  # Título do gráfico.
    plt.grid(axis='y', linestyle='--', alpha=0.7);  # Adiciona uma grade horizontal.
    plt.tight_layout();  # Ajusta o layout para evitar sobreposição de rótulos.
    plt.show();  # Exibe o gráfico.

    # Aviso importante para o usuário em um ambiente de notebook.
    print("\n-------------------------------------------------------------------");
    print("AVISO: O gráfico foi exibido. A execução do menu pode ter sido interrompida.");
    print("Caso isso tenha acontecido, execute a célula do 'MENU DE OPÇÕES' novamente.");
    print("-------------------------------------------------------------------");

    return False; # Retorna False para sinalizar a interrupção da execução do menu.

# --- Ponto de Execução do Carregamento ---
if 'caminho_do_arquivo' in locals() and caminho_do_arquivo:
    # Se um arquivo foi enviado na Célula 1, chama a função para carregar seus dados.
    dados_meteorologicos = carregar_dados(caminho_do_arquivo);
else:
    # Se a Célula 1 não foi executada com sucesso, informa o usuário.
    dados_meteorologicos = None;
    print("\nDados não foram carregados. Por favor, execute o Bloco 3 agora");

In [ ]:
# --- Bloco 3: Indicar Caminho do Arquivo ---

print("\n--- Upload de Arquivo ---");
# Variáveis globais que serão usadas nas outras células
dados_meteorologicos = None
caminho_do_arquivo_final = None

# --- Passo 1: Interação com o usuário para escolher a fonte ---
print("--- Fonte dos Dados ---")
print("Como você deseja fornecer o arquivo de dados?")
print("1) Fazer upload do meu computador agora (Colab)")
print("2) Usar um arquivo existente no ambiente (especificar caminho)")

if in_colab:
    escolha_fonte = input("Digite a opção (1 ou 2): ").strip()
else:
    # Se não estiver no Colab, força a opção 2 (especificar caminho)
    print("Detectado ambiente fora do Colab. Usando a opção 2 (especificar caminho).")
    escolha_fonte = '2'


# --- Passo 2: Execução baseada na escolha ---
if escolha_fonte == '1' and in_colab:
    print("\nPor favor, selecione o arquivo .csv para upload.")
    # Inicia o processo de upload de arquivos do seu computador
    uploaded = files.upload()

    if uploaded:
        # Pega o nome do arquivo que o usuário enviou
        caminho_do_arquivo_final = list(uploaded.keys())[0]
        print(f"\nSucesso! Arquivo '{caminho_do_arquivo_final}' foi enviado.")
    else:
        # Caso o usuário cancele a janela de upload
        print("\nERRO: Nenhum arquivo foi selecionado.")

elif escolha_fonte == '2':
    # Pede ao usuário que informe o caminho do arquivo
    caminho_personalizado = input("Por favor, digite o caminho completo do arquivo (ex: dados.csv ou /content/amostra/dados.csv): ").strip()

    # Verifica se o arquivo realmente existe no caminho especificado
    if os.path.exists(caminho_personalizado):
        caminho_do_arquivo_final = caminho_personalizado
        print(f"\nSucesso! Usando o arquivo em '{caminho_do_arquivo_final}'.")
    else:
        print(f"\nERRO: Arquivo não encontrado no caminho '{caminho_personalizado}'. Verifique o caminho e tente novamente.")

else:
    print("ERRO: Opção inválida. Por favor, execute esta célula novamente e escolha 1 ou 2.")

# --- Passo 3: Carregamento dos dados, se um caminho válido foi obtido ---
if caminho_do_arquivo_final:
    # Chama a função de carga (definida na Célula 3) para processar o arquivo
    dados_meteorologicos = carregar_dados(caminho_do_arquivo_final)

# Confirmação final para o usuário
if dados_meteorologicos:
    print("\nDados prontos para análise. Execute a célula do menu para começar.")
else:
    print("\nA carga de dados falhou. Não será possível continuar com as análises.")

def executar_menu():
    # Verifica se variável global 'dados_meteorologicos' foi criada e preenchida pela Célula 2
    if 'dados_meteorologicos' not in globals() or dados_meteorologicos is None:
        print("\nERRO CRÍTICO: Os dados não estão carregados.");
        print("Por favor, execute as Células 1 e 2 antes de rodar esta célula.");
        return; # Interrompe a execução da função se os dados não estiverem prontos.

In [ ]:
# --- Bloco 4: Menu Interativo e Execução ---

def executar_menu():
    # Verifica se variável global 'dados_meteorologicos' foi criada e preenchida pela Célula 2
    if 'dados_meteorologicos' not in globals() or dados_meteorologicos is None:
        print("\nERRO CRÍTICO: Os dados não estão carregados.");
        print("Por favor, execute os Blocos 1, 2 e 3 antes de rodar esta célula.");
        return; # Interrompe a execução da função se os dados não estiverem prontos.

    # Inicia um loop infinito que mantém o menu ativo até que o usuário decida sair.
    while True:
        # Imprime o cabeçalho do menu para uma interface clara e organizada.
        print("\n" + "="*25 + " MENU DE OPÇÕES " + "="*25);
        print("a) Visualizar dados por período");
        print("b) Ver mês(es) mais chuvoso(s)");
        print("c) Análise completa da temperatura mínima (irá parar o menu)");
        print("d) Sair do programa");

        # Captura a entrada do usuário e a converte para minúsculas.
        opcao = input("Escolha uma opção: ").lower();

        # Estrutura condicional para chamar a função apropriada com base na escolha do usuário.
        if opcao == 'a':
            visualizar_intervalo_dados(dados_meteorologicos);
        elif opcao == 'b':
            encontrar_mes_chuvoso(dados_meteorologicos);
        elif opcao == 'c':
            analisar_temperatura_minima(dados_meteorologicos);
        elif opcao == 'd':
            # Se o usuário digitar '0', exibe uma mensagem de despedida e encerra o loop.
            print("Obrigado por usar o sistema. Encerrando...");
            break;
        else:
            # Se a entrada não for nenhuma das opções válidas, informa o usuário sobre o erro.
            print("ERRO: Opção inválida. Por favor, escolha uma das opções do menu.");

# Ponto de entrada para este bloco.
executar_menu()